In [8]:
# what is it 
# a lookup table of shae (vocab_size, d_model), each of 8000 tokne
# ID maps to a learned d_model- dimensional vector 

import torch 
import math 
import torch.nn as nn

In [6]:
def sin_positional_encoding(max_seq_len, d_model):
    position = torch.arange(max_seq_len).unsqueeze(1) #shape (max, seq_len,1)
    i = torch.arange(0, d_model, 2) #gives [0,2,4,,D_model]
    div_term = torch.exp(i * (-math.log(10000.0) / d_model))     # (d_model/2,)
    angle = position * div_term
    pe = torch.zeros(max_seq_len, d_model)
    pe[:, 0::2] = torch.sin(angle)
    pe[:, 1::2] = torch.cos(angle)
    return pe 


In [9]:
pe = sin_positional_encoding(max_seq_len=100, d_model=16)
print(pe.shape)        
print(pe[0])           # position 0: sin(0)=0 for all even, cos(0)=1 for all odd -> should be [0,1,0,1,0,1,...]
print(pe[1, :4])       # position 1: check non-trivial sin/cos values

torch.Size([100, 16])
tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.])
tensor([0.8415, 0.5403, 0.3110, 0.9504])


In [11]:
class TokenAndPositionalEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        pe = sin_positional_encoding(max_seq_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, X):
        vector  = self.token_embedding(X)  # (bathch, seq_len, d_model
        scaled_vec = vector * torch.sqrt(self.d_model)
        seq_len = X.shape[1]
        pos_enc = self.pe[:seq_len]
        return scaled_vec +  pos_enc



    